In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline

# Install contextily if not already installed
import subprocess
subprocess.run(["pip", "install", "contextily", "--break-system-packages", "-q"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'contextily', '--break-system-packages', '-q'], returncode=0, stdout=b'', stderr=b'')

In [2]:
class NYCLayoverChatbot:
    """NYC Airport Layover Itinerary Recommendation Chatbot"""
    
    def __init__(self, dataset_path='nyc_attractions_final.csv'):
        """Initialize chatbot with attraction dataset"""
        
        # Load dataset
        self.df = pd.read_csv(dataset_path)
        
        # Track recommended attractions (to avoid duplicates)
        self.recommended_attractions = set()
        
        # Store last generated map for display after run()
        self.last_map = None
                
        # Budget hierarchy (for filtering)
        self.BUDGET_HIERARCHY = {'Low': 1, 'Medium': 2, 'High': 3}
        
        # Airport travel times by neighborhood (one-way, in minutes)
        self.AIRPORT_TRANSIT = {
            'JFK': {
                'Midtown': 60,
                'Chelsea': 65,
                'Upper East Side': 60,
                'Upper West Side': 70,
                'Manhattan Other': 65,
                'Greenwich Village': 65,
                'Downtown': 55,
                'Brooklyn': 45,
                'Bronx': 90
            },
            'LGA': {
                'Midtown': 35,
                'Chelsea': 40,
                'Upper East Side': 30,
                'Upper West Side': 45,
                'Manhattan Other': 40,
                'Greenwich Village': 45,
                'Downtown': 50,
                'Brooklyn': 50,
                'Bronx': 45
            },
            'EWR': {
                'Midtown': 50,
                'Chelsea': 45,
                'Upper East Side': 60,
                'Upper West Side': 55,
                'Manhattan Other': 55,
                'Greenwich Village': 50,
                'Downtown': 45,
                'Brooklyn': 55,
                'Bronx': 75
            }
        }
        
        # Inter-attraction travel times (minutes)
        self.ADJACENT_NEIGHBORHOODS = [
            ('Midtown', 'Chelsea'),
            ('Midtown', 'Upper East Side'),
            ('Midtown', 'Upper West Side'),
            ('Chelsea', 'Greenwich Village'),
            ('Greenwich Village', 'Downtown')
        ]
        
        self.MANHATTAN_NEIGHBORHOODS = [
            'Midtown', 'Upper East Side', 'Upper West Side',
            'Chelsea', 'Greenwich Village', 'Manhattan Other', 'Downtown'
        ]
    
    def get_travel_time_between(self, neighborhood1, neighborhood2):
        """Get travel time between two neighborhoods (minutes)"""
        
        # Same neighborhood
        if neighborhood1 == neighborhood2:
            return 10
        
        # Adjacent Manhattan neighborhoods
        if ((neighborhood1, neighborhood2) in self.ADJACENT_NEIGHBORHOODS or
            (neighborhood2, neighborhood1) in self.ADJACENT_NEIGHBORHOODS):
            return 25
        
        # Both in Manhattan (not adjacent)
        if (neighborhood1 in self.MANHATTAN_NEIGHBORHOODS and
            neighborhood2 in self.MANHATTAN_NEIGHBORHOODS):
            return 35
        
        # Cross-borough or one outer borough
        return 45
    
    def calculate_itinerary_time(self, itinerary, airport):
        """
        Calculate total time needed for complete itinerary
        
        Returns: total hours (including all transit, NO buffer)
        """
        
        if len(itinerary) == 0:
            return 0
        
        total_minutes = 0
        
        # Airport → First attraction
        first_neighborhood = itinerary[0]['neighborhood']
        total_minutes += self.AIRPORT_TRANSIT[airport].get(first_neighborhood, 55)
        
        # Visit each attraction + travel to next
        for i, attraction in enumerate(itinerary):
            # Visit duration
            total_minutes += attraction['visit_duration_hrs'] * 60
            
            # Travel to next attraction (if not last)
            if i < len(itinerary) - 1:
                travel_time = self.get_travel_time_between(
                    attraction['neighborhood'],
                    itinerary[i + 1]['neighborhood']
                )
                total_minutes += travel_time
        
        # Last attraction → Airport
        last_neighborhood = itinerary[-1]['neighborhood']
        total_minutes += self.AIRPORT_TRANSIT[airport].get(last_neighborhood, 55)
        
        # Convert to hours (NO buffer added)
        total_hours = total_minutes / 60
        
        return total_hours
    
    
    def score_attraction(self, attraction, category_rank, user_budget, current_itinerary):
        """
        Score an attraction based on multiple factors
        
        Returns: int score (higher is better, -999 means excluded)
        """
        
        score = 0
        
        # 1. CATEGORY MATCH (by selection order)
        category_points = {1: 100, 2: 80, 3: 60, 4: 40, 5: 20}
        score += category_points.get(category_rank, max(0, 120 - category_rank * 20))
        
        # 2. BUDGET FIT
        user_tier = self.BUDGET_HIERARCHY[user_budget]
        attraction_tier = self.BUDGET_HIERARCHY[attraction['budget_level']]
        
        if attraction_tier > user_tier:
            return -999  # Exclude - above budget
        elif attraction_tier == user_tier:
            score += 50  # Exact match
        else:
            score += 25  # Below budget (good value)
        
        # 3. PROXIMITY BONUS (encourage closer attractions)
        proximity_bonus = {
            'Midtown': 20,
            'Chelsea': 18,
            'Upper East Side': 15,
            'Upper West Side': 15,
            'Manhattan Other': 12,
            'Greenwich Village': 10,
            'Downtown': 8,
            'Brooklyn': 0,
            'Bronx': 0
        }
        score += proximity_bonus.get(attraction['neighborhood'], 0)
        
        # 4. FAME BONUS (only the most iconic NYC attractions)
        famous_attractions = [
            'EMPIRE STATE',           # Empire State Building
            'STATUE OF LIBERTY',      # Statue of Liberty
            'METROPOLITAN MUSEUM',    # The Met
            'MOMA',                   # Museum of Modern Art
            'AMERICAN MUSEUM OF NATURAL HISTORY',  # AMNH
            'GRAND CENTRAL TERMINAL', # Grand Central
            'TIMES SQUARE',           # Times Square
            '9/11 MEMORIAL',          # 9/11 Memorial
            'BROOKLYN BRIDGE PARK MAIN STREET',        # Brooklyn Bridge
            'CENTRAL PARK',           # Central Park (when it's an attraction, not just "mall")
            'HIGH LINE',              # High Line
            'ROCKEFELLER CENTER',     # Rockefeller Center
            'CHELSEA MARKET'          # Chelsea Market
        ]

        if any(famous in attraction['name'].upper() for famous in famous_attractions):
            score += 200  # Massive boost for world-famous must-see attractions
                
        # 5. DURATION BONUS (substantial attractions)
        duration_hours = attraction['visit_duration_hrs']
        if duration_hours >= 2.5:
            score += 80  # Major attractions
        elif duration_hours >= 1.5:
            score += 40  # Medium attractions
        elif duration_hours >= 1.0:
            score += 20  # Quick stops
        
        # 6. DIVERSITY BONUS (encourage different neighborhoods)
        if len(current_itinerary) > 0:
            last_neighborhood = current_itinerary[-1]['neighborhood']
            if attraction['neighborhood'] == last_neighborhood:
                score -= 20  # Penalty for same neighborhood as previous
                
        return score
    
    def get_recommendation(self, category, category_rank, user_budget, 
                          time_preference, current_itinerary, available_hours, airport):
        """
        Get best recommendation for a category that fits constraints
        
        Returns: dict with attraction data, or None if nothing fits
        """
        
        # Filter by category
        candidates = self.df[self.df['category'] == category].copy()
        
        # Filter by time preference
        if time_preference == 'Daytime':
            candidates = candidates[candidates['slot_daytime'] == 1]
        elif time_preference == 'Evening':
            candidates = candidates[candidates['slot_evening'] == 1]
        # Flexible = no filtering
        
        # Exclude already recommended
        candidates = candidates[~candidates['name'].isin(self.recommended_attractions)]
        
        if len(candidates) == 0:
            return None
        
        # Score each candidate
        candidates['score'] = candidates.apply(
            lambda row: self.score_attraction(row, category_rank, user_budget, current_itinerary),
            axis=1
        )
        
        # Filter out excluded attractions (score = -999)
        candidates = candidates[candidates['score'] > -999]
        
        if len(candidates) == 0:
            return None
        
        # Sort by score
        candidates = candidates.sort_values('score', ascending=False)
        
        # Try each candidate (highest score first) until one fits time-wise
        for idx, candidate in candidates.iterrows():
            test_itinerary = current_itinerary + [candidate]
            total_time = self.calculate_itinerary_time(test_itinerary, airport)
            
            if total_time <= available_hours:
                # Mark as recommended
                self.recommended_attractions.add(candidate['name'])
                return candidate
        
        # No candidates fit in available time
        return None
    
    def calculate_total_cost(self, itinerary):
        """Calculate total cost of itinerary"""
        
        total = 0
        
        for attraction in itinerary:
            cost = attraction['entry_cost']
            
            if cost == 'FREE':
                continue
            
            # Parse cost (handle ranges like "$20-40")
            cost_str = str(cost).replace('$', '').strip()
            
            if '-' in cost_str:
                # Take max of range
                parts = cost_str.split('-')
                try:
                    total += float(parts[1])
                except:
                    pass
            else:
                try:
                    total += float(cost_str)
                except:
                    pass
        
        return total
    
    def generate_itinerary_map(self, itinerary, airport):
        """Plot selected attractions over a real NYC map using contextily."""
        
        if len(itinerary) == 0:
            return
        
        import contextily as ctx
        from pyproj import Transformer
        
        points = []
        for i, attraction in enumerate(itinerary, start=1):
            points.append({
                'order': i,
                'name': attraction['name'],
                'neighborhood': attraction['neighborhood'],
                'lat': float(attraction['latitude']),
                'lon': float(attraction['longitude'])
            })
        
        # Convert WGS84 (lat/lon) to Web Mercator (metres) for contextily
        transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
        for p in points:
            p['x'], p['y'] = transformer.transform(p['lon'], p['lat'])
        
        xs = [p['x'] for p in points]
        ys = [p['y'] for p in points]
        pad_x = 2200
        pad_y = 2200
        
        fig, ax = plt.subplots(figsize=(11, 10))
        
        # Set bounds BEFORE adding basemap
        ax.set_xlim(min(xs) - pad_x, max(xs) + pad_x)
        ax.set_ylim(min(ys) - pad_y, max(ys) + pad_y)
        
        # Draw map tiles
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=13)
        
        # Draw directional arrows between stops
        for i in range(len(points) - 1):
            ax.annotate(
                '',
                xy=(points[i+1]['x'], points[i+1]['y']),
                xytext=(points[i]['x'], points[i]['y']),
                arrowprops=dict(
                    arrowstyle='->', color='royalblue',
                    lw=2.5, mutation_scale=35
                ),
                zorder=2
            )
        
        # Draw markers
        ax.scatter(xs, ys, s=450, color='royalblue', zorder=4, edgecolors='white', linewidths=2.5)
        
        # Stop numbers inside markers
        for p in points:
            ax.text(p['x'], p['y'], str(p['order']),
                    color='white', fontsize=12, fontweight='bold',
                    ha='center', va='center', zorder=5)
        
        # Name labels with white background box
        for p in points:
            ax.annotate(
                p['name'],
                xy=(p['x'], p['y']),
                xytext=(0, -26),
                textcoords='offset points',
                fontsize=9,
                color='#111111',
                ha='center',
                va='top',
                bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#cccccc', lw=0.5),
                zorder=6
            )
        
        # Transit times from this airport per neighborhood
        transit_times = self.AIRPORT_TRANSIT[airport]
        first_transit = transit_times.get(points[0]['neighborhood'], 55)
        last_transit  = transit_times.get(points[-1]['neighborhood'], 55)

        # Arrow pointing OUT from top-left corner toward airport (first stop)
        ax.annotate(
            f'  ✈ From {airport}\n  ~{first_transit} min',
            xy=(points[0]['x'], points[0]['y']),
            xytext=(points[0]['x'] - pad_x * 0.6, points[0]['y'] + pad_y * 0.6),
            fontsize=8.5, color='#444444',
            ha='left', va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#aaaaaa', lw=0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                            mutation_scale=20, linestyle='dashed'),
            zorder=6
        )

        # Arrow pointing OUT from last stop toward airport
        ax.annotate(
            f'  ✈ Back to {airport}\n  ~{last_transit} min',
            xy=(points[-1]['x'], points[-1]['y']),
            xytext=(points[-1]['x'] + pad_x * 0.6, points[-1]['y'] - pad_y * 0.6),
            fontsize=8.5, color='#444444',
            ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.85, ec='#aaaaaa', lw=0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                            mutation_scale=20, linestyle='dashed'),
            zorder=6
        )

        ax.set_axis_off()
        ax.set_title(f'Your NYC Layover Itinerary  —  {airport} Airport',
                     fontsize=15, pad=14, fontweight='500')
        plt.tight_layout()
        plt.savefig('nyc_itinerary_map.png', dpi=150, bbox_inches='tight')
        plt.close()
        display(Image('nyc_itinerary_map.png'))
    
    def run(self):
        """Main chatbot interaction"""
        
        print("NYC AIRPORT LAYOVER ITINERARY CHATBOT")
        print("\nWelcome! I'll help you plan your NYC layover.\n")
        
        # Reset for new session
        self.recommended_attractions = set()
        
        # STEP 1: Get available time
        
        print("How many hours do you have available?")
        print("(From when you can leave the airport until you need to be back)")
        
        while True:
            try:
                available_hours = float(input("\nEnter hours (e.g., 6, 8.5, 10): ").strip())
                if available_hours > 0:
                    break
                print("Please enter a positive number.")
            except:
                print("Invalid input. Please enter a number (e.g., 6, 8.5, 10).")
        
        print(f"\n✓ Available time: {available_hours} hours")
        
        # STEP 2: Get airport
        
        print("\nWhich airport are you travelling from?")
        print("  1. JFK (John F. Kennedy)")
        print("  2. LGA (LaGuardia)")
        print("  3. EWR (Newark)")
        
        while True:
            choice = input("\nEnter choice (1-3): ").strip()
            if choice in ['1', '2', '3']:
                break
            print("Invalid input. Please enter 1, 2, or 3.")
        
        airport_map = {'1': 'JFK', '2': 'LGA', '3': 'EWR'}
        airport = airport_map[choice]
        
        print(f"\n✓ Airport: {airport}")
        
        # STEP 3: Get time preference
        
        print("\nWhen will you be visiting attractions?")
        print("  1. Daytime (9am-6pm)")
        print("  2. Evening (6pm-12am)")
        print("  3. Flexible (any time)")
        
        while True:
            choice = input("\nEnter choice (1-3): ").strip()
            if choice in ['1', '2', '3']:
                break
            print("Invalid input. Please enter 1, 2, or 3.")
        
        time_map = {'1': 'Daytime', '2': 'Evening', '3': 'Flexible'}
        time_preference = time_map[choice]
        
        print(f"\n✓ Time preference: {time_preference}")
        
        # STEP 4: Get budget
        
        print("\nWhat is your budget?")
        print("  1. Low (FREE - $30 per attraction)")
        print("  2. Medium ($31 - $100 per attraction)")
        print("  3. High ($100+ per attraction)")
        
        while True:
            choice = input("\nEnter choice (1-3): ").strip()
            if choice in ['1', '2', '3']:
                break
            print("Invalid input. Please enter 1, 2, or 3.")
        
        budget_map = {'1': 'Low', '2': 'Medium', '3': 'High'}
        user_budget = budget_map[choice]
        
        print(f"\n✓ Budget: {user_budget}")
        
        # STEP 5: Get category selections
        
        categories = [
            'Food & Drink',
            'Historical & Cultural Sites, Landmarks & Viewpoints',
            'Museums & Galleries',
            'Nature & Adventure',
            'Shopping',
            'Shows & Nightlife'
        ]
        
        print("\nSelect categories you'd like to explore.")
        print("(You can choose the same category multiple times)\n")
        
        for i, cat in enumerate(categories, 1):
            print(f"  {i}. {cat}")
        
        print("\nHow many attractions would you like recommendations for?")
        while True:
            try:
                num_recs = int(input("Enter number (1-10): ").strip())
                if 1 <= num_recs <= 10:
                    break
                print("Please enter a number between 1 and 10.")
            except:
                print("Invalid input. Please enter a number.")
        
        selected_categories = []
        for i in range(num_recs):
            # After the first pick, confirm what was just chosen before prompting again
            if i > 0:
                print(f"\nChoice {i}: {selected_categories[i-1]}")

            ordinals = ['first', 'second', 'third', 'fourth', 'fifth',
                        'sixth', 'seventh', 'eighth', 'ninth', 'tenth']
            print(f"Please select {ordinals[i]} choice (enter a number 1-6):")

            while True:
                try:
                    choice = int(input("  → ").strip())
                    if 1 <= choice <= 6:
                        selected_categories.append(categories[choice - 1])
                        break
                    print("    Please enter a number between 1 and 6.")
                except:
                    print("    Invalid input. Please enter a number.")
        
        print(f"\nChoice {num_recs}: {selected_categories[-1]}")
        print(f"\n✓ Selected {len(selected_categories)} categories")
        
        # STEP 6: Build itinerary
        
        print("BUILDING YOUR PERSONALIZED ITINERARY")
        
        itinerary = []
        
        for rank, category in enumerate(selected_categories, 1):
            recommendation = self.get_recommendation(
                category, rank, user_budget, time_preference,
                itinerary, available_hours, airport
            )
            
            if recommendation is None:
                print(f"{rank}. [{category}]")
                
                # Determine why no recommendation
                available = self.df[self.df['category'] == category]
                
                if time_preference == 'Daytime':
                    available = available[available['slot_daytime'] == 1]
                elif time_preference == 'Evening':
                    available = available[available['slot_evening'] == 1]
                
                if len(available) == 0:
                    print(f"     No {category} attractions available during {time_preference}")
                else:
                    print(f"     No {category} attractions fit in remaining time\n")
            else:
                itinerary.append(recommendation)
                
                print(f"{rank}. {recommendation['name']} - {category}")
                print(f"   Cost: {recommendation['entry_cost']:15} | " +
                      f"Duration: {recommendation['visit_duration_hrs']:.1f} hours | " +
                      f"Neighborhood: {recommendation['neighborhood']}")
                print()
        
        # STEP 7: Display summary
        
        if len(itinerary) > 0:
            total_time = self.calculate_itinerary_time(itinerary, airport)
            total_cost = self.calculate_total_cost(itinerary)
            
            print("ITINERARY SUMMARY")
            
            print(f"\nTotal attractions:         {len(itinerary)}")
            print(f"Total time needed:         {total_time:.1f} hours (visits + all transit)")
            print(f"Available time:            {available_hours:.1f} hours")
            print(f"Estimated total cost:      ${total_cost:.2f}")
            
            # Check if itinerary fits
            if total_time > available_hours:
                print(f"\n WARNING: This itinerary needs {total_time:.1f} hours but you only have {available_hours:.1f} hours!")
                print("   Consider removing one attraction.")
            elif total_time <= available_hours * 0.9:
                print(f"\n This itinerary fits comfortably in your {available_hours} hour window!")
            else:
                print(f"\n✓ This itinerary fits your {available_hours} hour window.")
            
            print("IMPORTANT REMINDERS")
            
            print("\n Before you go:")
            print("  • Verify opening hours - they may vary by day/season")
            print("  • Some attractions require advance ticket reservations")
            print("  • Allow extra time for security lines and unexpected delays")
            print("  • Check weather conditions for outdoor attractions")
                        # Generate map visualization
            self.generate_itinerary_map(itinerary, airport)
            
        else:
            
            print("\n Unable to find attractions matching all your criteria.")
            print("\nSuggestions:")
            print("  • Try selecting 'Flexible' for time preference")
            print("  • Increase your budget level")
            print("  • Select fewer attractions")
            print("  • Choose different categories")
        
        print("Thank you for using NYC Airport Layover Chatbot!")
        print("\nSafe travels!\n")

In [6]:
chatbot = NYCLayoverChatbot()
chatbot.run()

NYC AIRPORT LAYOVER ITINERARY CHATBOT

Welcome! I'll help you plan your NYC layover.

How many hours do you have available?
(From when you can leave the airport until you need to be back)
